# Bronze: raw PDFs

Loads the raw FinanceBench PDFs from the Unity Catalog volume into the Delta table `bronze_pdfs`, as is. No parsing or transformation happens here.

```
/Volumes/<catalog>/<schema>/docs/pdfs/*.pdf  -->  <catalog>.<schema>.bronze_pdfs
```

**Incremental.** The notebook uses Auto Loader with a checkpoint, so each run loads only the PDFs it has not seen before. The first run loads all of them, and a rerun with nothing new loads 0 rows. It runs as the `bronze_pdfs` task of the `ingest_financebench` job, after the `download` task.

**Table columns**

| Column | Meaning |
| --- | --- |
| `path` | Full volume path |
| `modificationTime` | File modification time |
| `length` | File size in bytes |
| `content` | Raw PDF bytes |

## 1. Imports

No imports are needed. `spark` and `dbutils` are provided by the Databricks notebook runtime.

## 2. Parameters

The job passes `catalog` and `bronze_schema` from the bundle variables. The defaults match [databricks.yml](../databricks.yml), so the notebook also runs on its own.

In [ ]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("bronze_schema", "finance_bench_bronze")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("bronze_schema")

volume = f"/Volumes/{catalog}/{schema}/docs"
table = f"{catalog}.{schema}.bronze_pdfs"
checkpoint = f"{volume}/_checkpoints/bronze_pdfs"

## 3. Read the PDFs with Auto Loader

Reads every `*.pdf` under `pdfs/` as a stream in the `binaryFile` format, which gives one row per file with its raw bytes. Auto Loader records the files it has processed in the checkpoint, so only new files come through on later runs.

In [ ]:
df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "binaryFile")
    .option("pathGlobFilter", "*.pdf")
    .load(f"{volume}/pdfs")
)

## 4. Write to the bronze table

Appends the new rows to `bronze_pdfs`.

- `trigger(availableNow=True)` processes everything new and then stops, so this behaves like a batch task and does not run forever.
- `awaitTermination()` makes the task wait for the load to finish. If the load fails, the task fails.
- The checkpoint lives in the volume at `_checkpoints/bronze_pdfs`.

In [ ]:
(
    df.writeStream
    .option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .toTable(table)
    .awaitTermination()
)

## 5. Check the result

Counts the rows now in the table. After the first run this should equal the number of PDFs in the volume (368 for FinanceBench).

In [ ]:
print(f"{table}: {spark.table(table).count()} rows")

## Notes

- Files are tracked by path, so a PDF that changes upstream under the same name is not reloaded.
- **Full rebuild:** drop `bronze_pdfs`, delete the checkpoint folder (`/Volumes/<catalog>/<schema>/docs/_checkpoints/bronze_pdfs`) and rerun. Dropping only the table leaves the checkpoint, so nothing would be reloaded.
- No file arrival trigger is needed, because the `download` task writes the PDFs in the same job run.
- The service principal needs `CREATE TABLE` on the schema and write access to the volume. See [doc/setup.md](../doc/setup.md#one-time-setup).